# VRM + CAM: ResNet32x4 to ResNet8x4 —28px 3 pass(es)

This is the **published-code VRM variant** (VISION-SJTU/VRM@56d02ce), wrapped in our
CAM curriculum. The code omits the paper's joint-entropy unreliable-edge mask; no
unverified pruning rule is added. Losses, reductions, logit normalization and nested
weights match that code exactly. No independent full/crop KD or extra KL loss is added.

| Size | Epochs | Stride | Passes | Two VRM inputs |
|---|---|---|---|---|
|16|1–48|8|3|Weak full image + weak CAM crop|
|20|49–96|8|3|Weak full image + weak CAM crop|
|24|97–144|8|3|Weak full image + weak CAM crop|
|28|145–192|2|3|Weak full image + weak CAM crop|
|32|193–240|—|1|**Original VRM weak + strong32px views**|

28px has nine grid positions; top3 are cached. One pass uses top1; three passes use
top1/top2/top3. Scores are mean ground-truth-class CAM, not teacher confidence.
At32px, original RandomCrop32/padding4/flip + customRandAugment(2,10)/Cutout and
original normalization are used, including original constant padding. Both32px student
forwards update BN as official VRM does. At smaller sizes crop forwards preserve BN
running statistics, while the full branch updates them (EP-KD policy).

Batch64, FP32, SGD LR.05/momentum.9/weight decay.0005,240epochs. This integration uses
the EP-KD LR cuts48/144/192/210, not the paper's baseline150/180/210 schedule.
Seeds0/1/2 run once each, sequentially on GPU. Change GPU to select another GPU.
Option2 has more optimizer steps at28px; this is not an equal-compute comparison.

**Kaggle:** enable Internet and a GPU, then Run All. Setup clones the published code
from GitHub at a pinned commit; no runtime Dataset or ZIP is needed.
Teacher/original CIFAR can be attached or downloaded.
Set SMOKE_TEST=True for a short5stage stop/resume/export test before full training.
Logging, plots and resumable checkpoints are saved under OUTPUT. Output paths distinguish
the two options and smoke runs. GPU throughput and full benchmark accuracy are unmeasured.


In [ ]:
from pathlib import Path
REPO = Path("/kaggle/working/mdistiller_vrm_github")
REPO_URL = "https://github.com/ahmad903501/tgsc-mdistiller.git"
REVISION = '49113223d7c98422888321fbad8ff410953e4a46'
REQUIRED_SOURCE_FILES = ["tools/run_epkd.py", "mdistiller/epkd/runtime.py", "mdistiller/distillers/VRM.py", "mdistiller/epkd/vrm_augment.py"]
DATA_ROOT = Path("/kaggle/working/data")
OUTPUT = Path("/kaggle/working/vrm_cam_28p3")
TEACHER = "resnet32x4"
TEACHER_CHECKPOINT = None  # Discover an attached teacher or download official weights.
CONFIG_NAME = 'vrm_cam_resnet32x4_to_resnet8x4_28p3.json'
GPU = 0
SEEDS = [0,1,2]
EVAL_EVERY = 5
TEST_DATA = None
SMOKE_TEST = False  # True:128images,1epoch/size,first seed only; retain actual pass counts.


In [ ]:
from pathlib import Path
import json, subprocess, sys

# Clone the published implementation; no runtime ZIP/Dataset is needed.
def complete(root):
    return all((root/path).is_file() for path in REQUIRED_SOURCE_FILES)

cloned_now = False
if not (REPO/".git").exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError("REPO contains a non-Git runtime. Select a new writable checkout directory.")
    REPO.parent.mkdir(parents=True,exist_ok=True)
    subprocess.run(["git","clone","--quiet","--no-checkout",REPO_URL,str(REPO)],check=True)
    cloned_now = True
dirty = subprocess.check_output(["git","-C",str(REPO),"status","--porcelain","--untracked-files=no"],text=True)
if dirty.strip() and not cloned_now:
    raise RuntimeError("Tracked source changes in REPO; select a fresh directory instead of overwriting them.")
available = subprocess.run(["git","-C",str(REPO),"rev-parse","--verify",REVISION+"^{commit}"],capture_output=True)
if available.returncode:
    subprocess.run(["git","-C",str(REPO),"fetch","--quiet","origin",REVISION],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--quiet","--detach",REVISION],check=True)
actual = subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
if len(REVISION) != 40 or actual != REVISION:
    raise RuntimeError("Notebook must select the exact published40-character source commit.")
if not complete(REPO):
    raise RuntimeError("Selected source commit does not contain the required implementation.")
print("Source:",REPO_URL,"commit:",actual)

subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(REPO/"requirements-tgsc.txt")],check=True)
import torch
assert torch.cuda.is_available(), "Enable GPU; this notebook does not silently train on CPU."
print("Runtime:",REPO,"GPU:",torch.cuda.get_device_name(GPU))
OUTPUT.mkdir(parents=True,exist_ok=True)


In [ ]:
# Reuse only the teacher from an attached old Dataset; do not read its crop bank.
if TEACHER_CHECKPOINT is None:
    matches = [p for p in Path("/kaggle/input").rglob("ckpt_epoch_240.pth") if p.parent.name == TEACHER+"_vanilla"]
    if len(matches) > 1:
        raise RuntimeError("Multiple teacher copies found; set TEACHER_CHECKPOINT explicitly.")
    if matches:
        TEACHER_CHECKPOINT = matches[0]
    else:
        subprocess.run([sys.executable, str(REPO/"tools/download_epkd_teacher.py"), "--teacher", TEACHER,
                        "--output-dir", str(OUTPUT/"teacher_weights")], check=True)
        TEACHER_CHECKPOINT = OUTPUT/"teacher_weights/cifar_teachers"/(TEACHER+"_vanilla")/"ckpt_epoch_240.pth"
assert Path(TEACHER_CHECKPOINT).is_file()
# Set DATA_ROOT to a folder containing cifar-100-python/, including an attached read-only Dataset.
# Otherwise torchvision downloads the original archive once, shared by all methods/seeds here.
from torchvision.datasets import CIFAR100
if not (DATA_ROOT/"cifar-100-python/train").exists():
    CIFAR100(str(DATA_ROOT), train=True, download=True)
print("Teacher:", TEACHER_CHECKPOINT, "Original CIFAR:", DATA_ROOT)


In [ ]:
# Explicit full-run launcher: no synthetic data, no stored crop banks.
def run(command, config, seed, output, extra=()):
    cmd = [sys.executable, str(REPO/"tools/run_epkd.py"), command,
           "--config", str(config), "--teacher-checkpoint", str(TEACHER_CHECKPOINT),
           "--data-root", str(DATA_ROOT), "--output", str(output), "--gpu", str(GPU),
           "--seed", str(seed), "--eval-every", str(EVAL_EVERY), "--download", *extra]
    if TEST_DATA:
        cmd.extend(["--test-data", str(TEST_DATA)])
    subprocess.run(cmd, cwd=REPO, check=True)

config = REPO/"configs/epkd/cifar100"/CONFIG_NAME
if SMOKE_TEST:
    raw = json.loads(config.read_text())
    raw["data"]["stages"] = [[r,1] for r in (16,20,24,28,32)]
    raw["train"]["epochs"] = 5
    config = OUTPUT/(config.stem+"_SMOKE.json")
    config.write_text(json.dumps(raw,indent=2))
for seed in (SEEDS[:1] if SMOKE_TEST else SEEDS):
    output = OUTPUT/(config.stem+f"_seed{seed}")
    latest = output/"latest.pt"
    run("check",config,seed,output,["--limit","16"])
    if SMOKE_TEST:
        if not latest.exists():
            run("train",config,seed,output,["--limit","128","--stop-after-epoch","2"])
        run("train",config,seed,output,["--limit","128","--resume",str(latest)])
    else:
        run("train",config,seed,output,["--resume",str(latest)] if latest.exists() else [])
    export = output/("inference_latest.pt" if SMOKE_TEST else "inference_best.pt")
    run("evaluate",config,seed,output,["--export",str(export),*(["--limit","128"] if SMOKE_TEST else [])])
    if SMOKE_TEST:
        saved = torch.load(latest,map_location="cpu",weights_only=True)
        history = saved["metrics_history"]
        assert [r["resolution"] for r in history] == [16,20,24,28,32]
        assert history[3]["passes"] == json.loads(config.read_text())["epkd"]["passes"][3]
        assert history[-1]["second_view"] == "vrm_original_strong32"
        assert all(set(r["train"]) == {"pair.loss_ce","pair.loss_isv","pair.loss_icv","loss_total"} for r in history)
        print("PASS: all stages,paired loss,28px pass count,original32px augmentation,resume and export")
    from IPython.display import display,Image
    display(Image(filename=str(output/"training_curves.png")))
